# Lab 5 — Ensemble the Churn Model
**SDA-AIE-111 · Day 3 · Hour 5 · 50 minutes · pairs**

**Objective.** Beat the logistic champion with forest and XGBoost **under the shared harness**; produce permutation importance and one partial-dependence reading; update the champion/challenger table with an argued recommendation.


In [ ]:
# --- Course setup (identical in every lab) -----------------------------------
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

def find_repo_root() -> Path:
    """Locate the course repo root by looking for data/manafeth_customers.parquet."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "data" / "manafeth_customers.parquet").exists():
            return cand
    raise FileNotFoundError(
        "Course data not found — copy the Manafeth data package files into <repo>/data/")

ROOT = find_repo_root()
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)

In [ ]:
from sklearn.model_selection import train_test_split
from manafeth.features import (build_preprocessor, build_feature_table,
                               MODEL_NUM_COLS, CAT_COLS, LEAKY)
from manafeth.evaluation import CV, evaluate, recall_at_budget, precision_at_budget

customers = pd.read_parquet(DATA / "manafeth_customers.parquet")
orders = pd.read_parquet(DATA / "manafeth_orders.parquet")
full = build_feature_table(customers, orders, pd.Timestamp("2025-11-01"))

X_all = full.drop(columns=["churned_30d", "customer_id", *LEAKY])
y_all = full["churned_30d"]
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=RANDOM_STATE)
print("train:", X_train.shape, "| churn:", round(y_train.mean(), 3))

## Task 1 — Random forest: strong defaults, honest OOB *(10 min)*

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

def make_pipe(model):
    return Pipeline([("prep", build_preprocessor(MODEL_NUM_COLS, CAT_COLS)), ("clf", model)])

# TODO: RandomForestClassifier(n_estimators=300, max_features="sqrt", min_samples_leaf=5,
#       class_weight="balanced", oob_score=True) inside the pipeline; run evaluate()
# TODO: fit once and print the OOB score as a cross-check


## Task 2 — XGBoost with early stopping *(15 min)*

Low learning rate; let early stopping choose the length. The eval set is carved from **train** and kept apart from reported-score folds — reported scores still come from the shared harness.


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split as tts

# TODO: carve a 15% early-stopping eval set from TRAIN (stratified)
# TODO: fit XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=4,
#       subsample=0.8, colsample_bytree=0.8, scale_pos_weight=<neg/pos ratio>,
#       eval_metric="aucpr", early_stopping_rounds=50) and read best_iteration
# TODO: rebuild as a pipeline with the observed n_estimators; report via evaluate()


## Task 3 — n_estimators: a flattening, not a peak *(5 min)*

In [ ]:
# TODO: forest PR-AUC at n_estimators in {50, 100, 300, 600} (3-fold is fine here)
# TODO: observe the flattening — why does adding trees not overfit a forest?


## Task 4 — Permutation importance vs impurity *(10 min)*

Teach the flaw before the tool: impurity importances inflate high-cardinality/continuous features. Permutation importance on held-out data is the honest reading — and a leakage detector (a feature that dominates suspiciously is a suspect).


In [ ]:
from sklearn.inspection import permutation_importance

# TODO: permutation_importance on the fitted XGBoost pipeline, scored on the
#       held-out eval slice (n_repeats=10, seeded); boxplot the top 8
# TODO: compare with xgb_pipe["clf"].feature_importances_ — note one disagreement


## Task 5 — Partial dependence: the curve the CRM team can act on *(5 min)*

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# TODO: partial dependence for days_since_last_order and basket_trend
# TODO: write the one-sentence CRM reading (where does risk accelerate?)


## Task 6 — Champion, challenger *(5 min)*

Update `project/CHAMPION.md`: the harness table (all models, same folds), your recommendation, and the challenger-retention sentence (the interpretable logistic stays in the room — it is the sanity check and the fallback).

| model | PR-AUC (mean ± std) | recall@20% |
|---|---|---|
| dummy | | |
| logreg+features | | |
| tree(depth=6) | | |
| random_forest | | |
| **xgboost** | | |

**Commit:** `feat(lab5): ensembles under shared harness; champion argued`
